## 1. Setup

In [1]:
import sys, os
# Adjust if this notebook lives somewhere other than the project root.
PROJECT_ROOT = os.path.abspath(".")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import dask.dataframe as dd

from config.config import config
from models.dask_utils import identify_categorical_columns

paths = config['paths']
features_cfg = config['features']

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 140)


## 2. Load the feature dataset

In [2]:
candidate_paths = [
    ("train_data", paths.sampled_train_data),
    ("feature_dataset", paths.feature_dataset),
]

ddf = None
source_used = None
for name, p in candidate_paths:
    try:
        ddf = dd.read_parquet(p)
        source_used = name
        print(f"Loaded '{name}' from: {p}")
        break
    except Exception as e:
        print(f"Could not read '{name}' ({p}): {e}")

if ddf is None:
    raise FileNotFoundError(
        "No parquet dataset found at any candidate path. Run the data-ingestion "
        "-> cleaning -> feature-engineering -> target-creation -> dataset-creation "
        "modules first (or point `candidate_paths` above at an existing file)."
    )

print(f"\nPartitions: {ddf.npartitions}")
print(f"Columns:    {len(ddf.columns)}")


Loaded 'train_data' from: D:/Projects/credit_risk_scoring/data/features_sampled/train_data.parquet

Partitions: 64
Columns:    64


## 3. Row count

In [3]:
n_rows = len(ddf)  # triggers a single count() pass across all partitions
print(f"Rows: {n_rows:,}")


Rows: 500,001


## 4. Dtype-based split 

In [4]:
dtype_categorical = [c for c, dt in ddf.dtypes.items()
                     if not pd.api.types.is_numeric_dtype(dt) and not pd.api.types.is_datetime64_any_dtype(dt)]
dtype_numeric = [c for c in ddf.columns if c not in dtype_categorical]

print(f"String/object dtype columns ({len(dtype_categorical)}):")
print(dtype_categorical)
print(f"\nNumeric dtype columns ({len(dtype_numeric)}):")
print(dtype_numeric)


String/object dtype columns (3):
['LOAN_SEQUENCE_NUMBER', 'MONTHLY_REPORTING_PERIOD', 'MSA']

Numeric dtype columns (61):
['target', 'CURRENT_ACTUAL_UPB', 'CURRENT_LOAN_DELINQUENCY_STATUS', 'LOAN_AGE', 'REMAINING_MONTHS_TO_LEGAL_MATURITY', 'MODIFICATION_FLAG', 'CURRENT_INTEREST_RATE', 'ELTV', 'INTEREST_BEARING_UPB', 'CREDIT_SCORE', 'FIRST_TIME_HOMEBUYER_FLAG', 'MI_PERCENTAGE', 'NUMBER_OF_UNITS', 'OCCUPANCY_STATUS', 'ORIGINAL_CLTV', 'ORIGINAL_DTI', 'ORIGINAL_UPB', 'ORIGINAL_LTV', 'ORIGINAL_INTEREST_RATE', 'CHANNEL', 'PPM_FLAG', 'PROPERTY_STATE', 'PROPERTY_TYPE', 'LOAN_PURPOSE', 'ORIGINAL_LOAN_TERM', 'NUMBER_OF_BORROWERS', 'observation_year', 'observation_month', 'observation_quarter', 'seasonality_sin', 'seasonality_cos', 'loan_age_squared', 'loan_age_cubic', 'remaining_term_pct', 'max_delinquency_3m', 'max_delinquency_6m', 'max_delinquency_12m', 'rolling_mean_delinquency_6m', 'num_delinquent_months_12m', 'consecutive_delinquent_months', 'months_since_last_delinquency', 'delinquency_tre

## 5. Cardinality per column (single-pass approximate `nunique`)

Uses Dask's `nunique_approx()` (HyperLogLog-based), which is a single bounded pass over the data — not a full shuffle/groupby like exact `nunique()` would require at 47M rows. Approximate counts are within a few percent, which is more than accurate enough for a categorical/numeric decision.

In [5]:
cardinality = {}
for c in ddf.columns:
    try:
        cardinality[c] = int(ddf[c].nunique_approx().compute())
    except Exception as e:
        cardinality[c] = None
        print(f"  [skipped] {c}: {e}")

cardinality_s = pd.Series(cardinality, name="approx_cardinality").sort_values()
cardinality_s.to_frame()


,approx_cardinality
dti_ltv_interaction,1
ORIGINAL_LTV,1
target,2
ever_modified,2
PPM_FLAG,2
NUMBER_OF_BORROWERS,3
LOAN_PURPOSE,3
MODIFICATION_FLAG,3
PROPERTY_TYPE,3
FIRST_TIME_HOMEBUYER_FLAG,3


## 6. Null rate per column (single reduction pass)

`isnull().mean()` is a single column-wise reduction across all partitions — bounded output (one fraction per column), never a row-wise materialization.

In [6]:
null_rate = ddf.isnull().mean().compute()
null_rate.name = "null_rate"
null_rate.sort_values(ascending=False).to_frame()


,null_rate
ORIGINAL_LTV,1.000000
MODIFICATION_FLAG,0.999978
ELTV,0.999772
MSA,0.342931
LOAN_PURPOSE,0.303563
PROPERTY_TYPE,0.197918
rolling_max_balance_6m,0.087492
rolling_avg_balance_6m,0.087492
rolling_min_balance_6m,0.087492
max_delinquency_3m,0.087492


## 7. Constant / near-constant detection

- **Numeric columns**: `min() == max()` (exact constant) via a single column-wise reduction — the same check now built into `models/logistic.py`.
- **All columns, low-cardinality subset only** (cardinality ≤ 50, from step 5): dominant-value share via `value_counts(normalize=True)`, to catch *near*-constant columns (e.g. 99.7% one value) that technically have >1 unique value but will still behave almost like a constant and contribute little signal while adding collinearity risk. This is intentionally restricted to the low-cardinality subset so it stays cheap — a `value_counts` pass on a genuinely high-cardinality numeric column would be expensive and isn't useful here anyway.

In [7]:
# Exact constant check (numeric columns only)
numeric_ddf = ddf[dtype_numeric]
mins = numeric_ddf.min().compute()
maxs = numeric_ddf.max().compute()
is_constant = (mins == maxs)
constant_numeric_cols = is_constant[is_constant].index.tolist()

print(f"Exactly-constant numeric columns ({len(constant_numeric_cols)}):")
print(constant_numeric_cols)


Exactly-constant numeric columns (1):
['dti_ltv_interaction']


In [8]:
# Dominant-value share for low-cardinality columns (categorical OR low-cardinality numeric)
LOW_CARD_THRESHOLD = 50
low_card_cols = [c for c, n in cardinality.items() if n is not None and n <= LOW_CARD_THRESHOLD]

dominant_share = {}
for c in low_card_cols:
    try:
        vc = ddf[c].value_counts(normalize=True, dropna=False).compute()
        dominant_share[c] = float(vc.iloc[0]) if len(vc) else np.nan
    except Exception as e:
        dominant_share[c] = np.nan
        print(f"  [skipped] {c}: {e}")

dominant_share_s = pd.Series(dominant_share, name="dominant_value_share").sort_values(ascending=False)
dominant_share_s.to_frame()


,dominant_value_share
dti_ltv_interaction,1.000000
ORIGINAL_LTV,1.000000
months_since_modification,0.999978
ever_modified,0.999978
PPM_FLAG,0.997872
target,0.984632
months_since_last_delinquency,0.967006
FIRST_TIME_HOMEBUYER_FLAG,0.909950
NUMBER_OF_BORROWERS,0.585031
LOAN_PURPOSE,0.318861


In [9]:
NEAR_CONSTANT_THRESHOLD = 0.995  # >99.5% of rows share one value
near_constant_cols = dominant_share_s[dominant_share_s > NEAR_CONSTANT_THRESHOLD].index.tolist()
print(f"Near-constant columns (dominant share > {NEAR_CONSTANT_THRESHOLD:.1%}), {len(near_constant_cols)}:")
print(near_constant_cols)


Near-constant columns (dominant share > 99.5%), 5:
['dti_ltv_interaction', 'ORIGINAL_LTV', 'months_since_modification', 'ever_modified', 'PPM_FLAG']


## 8. High-cardinality columns among string/object columns

Freddie Mac SFLLD has a few columns that are identifier-like rather than categorical in the modeling sense (loan numbers, servicer/seller names, postal codes) — `config.features.drop_features` already excludes the obvious ones (`LOAN_SEQUENCE_NUMBER`, `SELLER_NAME`, `SERVICER_NAME`, `POSTAL_CODE`). This cell double-checks nothing else in the string columns is secretly high-cardinality and would blow up one-hot/ordinal encoding if it slipped into `categorical_features`.

In [10]:
HIGH_CARDINALITY_THRESHOLD = 100  # a "real" categorical column shouldn't need more buckets than this for this dataset

high_card_object_cols = [
    c for c in dtype_categorical
    if cardinality.get(c) is not None and cardinality[c] > HIGH_CARDINALITY_THRESHOLD
]
print(f"High-cardinality string/object columns (>{HIGH_CARDINALITY_THRESHOLD} distinct values), {len(high_card_object_cols)}:")
for c in high_card_object_cols:
    print(f"  {c}: ~{cardinality[c]} distinct values")


High-cardinality string/object columns (>100 distinct values), 3:
  LOAN_SEQUENCE_NUMBER: ~245605 distinct values
  MONTHLY_REPORTING_PERIOD: ~203 distinct values
  MSA: ~453 distinct values


## 9. Compare the model's auto-detection heuristic against `config`'s current list

This reproduces the exact discrepancy that caused the earlier production bug: `identify_categorical_columns()` (the fallback used whenever a model isn't given an explicit list) vs. `config.features.categorical_features` (the explicit list `main.py` now passes to every model).

In [11]:
heuristic_categorical = identify_categorical_columns(ddf)
config_categorical = list(features_cfg.categorical_features)

only_in_heuristic = sorted(set(heuristic_categorical) - set(config_categorical))
only_in_config = sorted(set(config_categorical) - set(heuristic_categorical))

print(f"Heuristic flagged {len(heuristic_categorical)} columns as categorical.")
print(f"Config currently lists {len(config_categorical)} columns as categorical.")
print(f"\nFlagged by heuristic but NOT in config ({len(only_in_heuristic)}):")
print(only_in_heuristic)
print(f"\nIn config but NOT flagged by heuristic ({len(only_in_config)}):")
print(only_in_config)


Heuristic flagged 33 columns as categorical.
Config currently lists 11 columns as categorical.

Flagged by heuristic but NOT in config (23):
['ELTV', 'LOAN_SEQUENCE_NUMBER', 'MONTHLY_REPORTING_PERIOD', 'MSA', 'ORIGINAL_LTV', 'consecutive_delinquent_months', 'delinquency_trend_6m', 'dti_ltv_interaction', 'ever_modified', 'max_delinquency_12m', 'max_delinquency_3m', 'max_delinquency_6m', 'months_since_last_delinquency', 'months_since_modification', 'num_delinquent_months_12m', 'observation_month', 'observation_quarter', 'observation_year', 'origination_year', 'rate_change_since_origination', 'seasonality_cos', 'seasonality_sin', 'target']

In config but NOT flagged by heuristic (1):
['PROPERTY_STATE']


## 10. Consolidated column profile

One table, one row per column, everything above joined together — this is what you actually want to scan to make categorical/numeric/drop decisions.

In [12]:
profile = pd.DataFrame({"column": ddf.columns}).set_index("column")
profile["dtype"] = pd.Series({c: str(dt) for c, dt in ddf.dtypes.items()})
profile["is_object_dtype"] = profile.index.isin(dtype_categorical)
profile["approx_cardinality"] = cardinality_s
profile["null_rate"] = null_rate
profile["is_constant_numeric"] = profile.index.isin(constant_numeric_cols)
profile["dominant_value_share"] = dominant_share_s
profile["is_near_constant"] = profile.index.isin(near_constant_cols)
profile["in_config_categorical_features"] = profile.index.isin(config_categorical)
profile["flagged_by_heuristic"] = profile.index.isin(heuristic_categorical)
profile["is_high_cardinality_object"] = profile.index.isin(high_card_object_cols)
profile["in_config_drop_features"] = profile.index.isin(features_cfg.drop_features)

profile = profile.sort_values(["is_object_dtype", "approx_cardinality"], ascending=[False, True])
profile


,dtype,is_object_dtype,approx_cardinality,null_rate,is_constant_numeric,dominant_value_share,is_near_constant,in_config_categorical_features,flagged_by_heuristic,is_high_cardinality_object,in_config_drop_features
column,,,,,,,,,,,
MONTHLY_REPORTING_PERIOD,string,True,203,0.000000,False,NaN,False,False,True,True,True
MSA,string,True,453,0.342931,False,NaN,False,False,True,True,False
LOAN_SEQUENCE_NUMBER,string,True,245605,0.000000,False,NaN,False,False,True,True,True
ORIGINAL_LTV,int32,False,1,1.000000,False,1.000000,True,False,True,False,False
dti_ltv_interaction,float64,False,1,0.000000,True,1.000000,True,False,True,False,False
target,int32,False,2,0.000000,False,0.984632,False,False,True,False,False
PPM_FLAG,int32,False,2,0.000000,False,0.997872,True,True,True,False,False
ever_modified,int32,False,2,0.000000,False,0.999978,True,False,True,False,False
MODIFICATION_FLAG,int32,False,3,0.999978,False,0.000018,False,True,True,False,False


## 11. Columns that need a manual decision

Everything with `is_object_dtype == False` and `approx_cardinality` below a threshold is a **numeric** column that happens to have few unique values — this is exactly the set the auto-detection heuristic over-classified as categorical last time. Some of these genuinely ARE categorical in spirit (e.g. a status code stored as an int); most are counts/flags that should stay numeric. Review this list by hand and decide which belong in `categorical_features`.

In [13]:
REVIEW_THRESHOLD = 20  # matches the heuristic's own default threshold

review_candidates = profile[
    (~profile["is_object_dtype"]) & (profile["approx_cardinality"] <= REVIEW_THRESHOLD)
].copy()

print(f"{len(review_candidates)} numeric, low-cardinality columns to manually review:")
review_candidates[["approx_cardinality", "null_rate", "dominant_value_share", "in_config_categorical_features"]]


27 numeric, low-cardinality columns to manually review:


,approx_cardinality,null_rate,dominant_value_share,in_config_categorical_features
column,,,,
ORIGINAL_LTV,1,1.000000,1.000000,False
dti_ltv_interaction,1,0.000000,1.000000,False
target,2,0.000000,0.984632,False
PPM_FLAG,2,0.000000,0.997872,True
ever_modified,2,0.000000,0.999978,False
MODIFICATION_FLAG,3,0.999978,0.000018,True
FIRST_TIME_HOMEBUYER_FLAG,3,0.000446,0.909950,True
OCCUPANCY_STATUS,3,0.000000,0.044676,True
PROPERTY_TYPE,3,0.197918,0.009164,True


For each of these, a quick look at the *actual* distinct values (not just the count) makes the categorical-vs-numeric call easier — e.g. `observation_month` taking values 1–12 is clearly ordinal/numeric, while a status code taking values like `{0, 1, 2, 3, 6, 9, 'R'}` is clearly categorical.

In [14]:
for c in review_candidates.index:
    try:
        sample_vals = ddf[c].dropna().unique().compute()
        sample_vals = sorted(sample_vals.tolist())[:15]
    except Exception as e:
        sample_vals = f"<could not sample: {e}>"
    print(f"{c:45s} n~{cardinality.get(c)!s:>4}  values: {sample_vals}")


ORIGINAL_LTV                                  n~   1  values: []
dti_ltv_interaction                           n~   1  values: [0.0]
target                                        n~   2  values: [0, 1]
PPM_FLAG                                      n~   2  values: [0, 1]
ever_modified                                 n~   2  values: [0, 1]
MODIFICATION_FLAG                             n~   3  values: [1.0, 2.0]
FIRST_TIME_HOMEBUYER_FLAG                     n~   3  values: [0.0, 1.0]
OCCUPANCY_STATUS                              n~   3  values: [1, 2, 3]
PROPERTY_TYPE                                 n~   3  values: [1.0, 5.0]
LOAN_PURPOSE                                  n~   3  values: [1.0, 3.0]
NUMBER_OF_BORROWERS                           n~   3  values: [1.0, 2.0]
observation_quarter                           n~   4  values: [1, 2, 3, 4]
delinquency_trend_6m                          n~   4  values: [0, 1, 2, 3]
NUMBER_OF_UNITS                               n~   9  values: [1.0, 2.0, 

## 12. Save the full profile to disk

In [15]:
out_dir = paths.eda_dir
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "column_profile.csv")
profile.to_csv(out_path)
print(f"Saved column profile to: {out_path}")


Saved column profile to: D:/Projects/credit_risk_scoring/results/eda\column_profile.csv


## 13. Ready-to-paste `config/config.py` snippets

Two lists, printed as valid Python so you can paste them straight into `FeatureConfig` in `config/config.py`:

- **`categorical_features`**: starts from the dtype-ground-truth string columns (always safe to include) plus anything already in `config`'s current list that also survived the constant/near-constant/high-cardinality checks. It deliberately does **not** auto-add anything from the "needs manual review" list in section 11 — add those yourself after reading the printed sample values above, one at a time, so you don't reintroduce the over-classification bug in the other direction.
- **`excluded_features`** (new list, not yet in config): constant, near-constant, and high-cardinality-object columns — candidates to fold into `FeatureConfig.drop_features` so they're excluded from the design matrix entirely rather than merely relying on each model's constant-column guard.

In [16]:
recommended_categorical = sorted(
    set(dtype_categorical)
    | (set(config_categorical) - set(constant_numeric_cols) - set(near_constant_cols) - set(high_card_object_cols))
)

recommended_excluded = sorted(
    set(constant_numeric_cols) | set(near_constant_cols) | set(high_card_object_cols)
    - set(features_cfg.drop_features)
)

def print_list_literal(name, items):
    print(f"{name}: List[str] = field(default_factory=lambda: [")
    for item in items:
        print(f"    '{item}',")
    print("])")
    print()

print_list_literal("categorical_features", recommended_categorical)
print_list_literal("excluded_features  # NEW -- fold into drop_features", recommended_excluded)


categorical_features: List[str] = field(default_factory=lambda: [
    'CHANNEL',
    'CURRENT_LOAN_DELINQUENCY_STATUS',
    'FIRST_TIME_HOMEBUYER_FLAG',
    'LOAN_PURPOSE',
    'LOAN_SEQUENCE_NUMBER',
    'MODIFICATION_FLAG',
    'MONTHLY_REPORTING_PERIOD',
    'MSA',
    'NUMBER_OF_BORROWERS',
    'NUMBER_OF_UNITS',
    'OCCUPANCY_STATUS',
    'PROPERTY_STATE',
    'PROPERTY_TYPE',
])

excluded_features  # NEW -- fold into drop_features: List[str] = field(default_factory=lambda: [
    'MSA',
    'ORIGINAL_LTV',
    'PPM_FLAG',
    'dti_ltv_interaction',
    'ever_modified',
    'months_since_modification',
])

